# NN Training

All models use complex dataset with full detector degradation model for training.

Models:
- V0 (Baseline): Direct Image Target, L1 Loss, No Normalization
- V1 (BatchNorm): Direct Image Target, L1 Loss, BatchNorm
- V2 (Residual): Residual Target, L1 Loss, BatchNorm
- V3 (HybridLoss): Residual Target, BatchNorm, Hybrid Loss (0.7MAE + 0.3SSIM)

Adaptive stopping implemented:
- max = 200 epochs
- 20 epochs w/o improvement = stop
- min loss improvement = 1e-4

Save best models, settings and loss and metrics values for each epoch for comparison. Determine best model based on these.

In [ ]:
import os
from pathlib import Path
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from src.data_helpers import GeometricDataGenerator
from src.NNdeconvolution import DeconvolutionUNet_Baseline, DeconvolutionUNet_BatchNorm, DeconvolutionUNet_Residual
from src.ml_helpers import run_experiment
from src.plotting_helpers import plot_curves, validation_metrics_comparison
from src.testing_helpers import model_comparison_table

In [ ]:
# Device-agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# Create directory for outputs
save_dir = "nn_training"
save_dir = Path(save_dir).mkdir(parents=True, exist_ok=True)

In [ ]:
# Define dataloading constants
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count() - 1
IMG_SIZE = 256
OVERLAP = 0.5
DATA_SEED = 42 # using the same datasets for all of the models

# Load training data
num_train_samples = 8000
train_dataset = GeometricDataGenerator(
    data_loading=True,
    num_samples=num_train_samples,
    size=IMG_SIZE,
    overlap_set_share=OVERLAP,
    poisson_noise=True,
    full_detector_model=True,
    seed=DATA_SEED
)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

# Load validation data
num_validation_samples = 2000
validation_dataset = GeometricDataGenerator(
    data_loading=True,
    num_samples=num_validation_samples,
    size=IMG_SIZE,
    overlap_set_share=OVERLAP,
    poisson_noise=True,
    full_detector_model=True,
    seed=DATA_SEED
)
validation_dataloader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS
)

In [ ]:
# -------------------- BASELINE MODEL -------------------------
SEED = 42
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.manual_seed(SEED)

# Create model instance and move it to device
baseline_model = DeconvolutionUNet_Baseline().to(device)
# Get number of trainable params
num_trainable_params = sum(
    p.numel() for p in baseline_model.parameters() if p.requires_grad
)
print(f"Number of trainable parameters for model: {num_trainable_params}")

# Define loss function
MAE_SHARE = 1.0
EDGE_SHARE = 0.0
SSIM_SHARE = 0.0
MSE_SHARE = 0.0

# Define optimizer
LEARNING_RATE = 1e-4

# Define scheduler
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 3

# Early Stopping
MAX_EPOCHS = 200
PATIENCE = 15
MIN_DELTA = 1e-4

# Saving
PATH = "nn_training/models"
FILENAME = "nn_baseline_model"

df_metrics_baseline, df_about_baseline = run_experiment(
    model=baseline_model, seed=SEED, overlap=0.5,
    train_dataloader=train_dataloader, validation_dataloader=validation_dataloader, batch_size=BATCH_SIZE,
    mae_share=MAE_SHARE, edge_share=EDGE_SHARE, ssim_share=SSIM_SHARE, mse_share=MSE_SHARE,
    learning_rate=LEARNING_RATE, scheduler_factor=SCHEDULER_FACTOR,scheduler_patience=SCHEDULER_PATIENCE,
    max_epochs=MAX_EPOCHS, patience=PATIENCE, min_delta=MIN_DELTA, do_early_stopping=True,
    device=device, path=PATH, filename=FILENAME
)

In [ ]:
epochs_range = range(len(df_metrics_baseline["train_loss_vals"]))
plot_curves(epochs_range, df_metrics_baseline["train_loss_vals"], df_metrics_baseline["validation_loss_vals"], var_to_plot="Loss", filename="nn_training/loss_curves_baseline.png")
plot_curves(epochs_range, df_metrics_baseline["train_mae_vals"], df_metrics_baseline["validation_mae_vals"], var_to_plot="MAE", filename="nn_training/mae_curves_baseline.png")
plot_curves(epochs_range, df_metrics_baseline["train_psnr_vals"], df_metrics_baseline["validation_psnr_vals"], var_to_plot="PSNR", filename="nn_training/psnr_curves_baseline.png")
plot_curves(epochs_range, df_metrics_baseline["train_ssim_vals"], df_metrics_baseline["validation_ssim_vals"], var_to_plot="SSIM", filename="nn_training/ssim_curves_baseline.png")

In [ ]:
# -------------------- BASELINE MODEL + BATCH NORM -------------------------
SEED = 42
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.manual_seed(SEED)

# Create model instance and move it to device
batchnorm_model = DeconvolutionUNet_BatchNorm().to(device)
# Get number of trainable params
num_trainable_params = sum(
    p.numel() for p in batchnorm_model.parameters() if p.requires_grad
)
print(f"Number of trainable parameters for model: {num_trainable_params}")

# Define loss function
MAE_SHARE = 1.0
EDGE_SHARE = 0.0
SSIM_SHARE = 0.0
MSE_SHARE = 0.0

# Define optimizer
LEARNING_RATE = 1e-4

# Define scheduler
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 3

# Early Stopping
MAX_EPOCHS = 200
PATIENCE = 15
MIN_DELTA = 1e-4

# Saving
PATH = "nn_training/models"
FILENAME = "nn_batchnorm_model"

df_metrics_batchnorm, df_about_batchnorm = run_experiment(
    model=batchnorm_model, seed=SEED, overlap=0.5,
    train_dataloader=train_dataloader, validation_dataloader=validation_dataloader, batch_size=BATCH_SIZE,
    mae_share=MAE_SHARE, edge_share=EDGE_SHARE, ssim_share=SSIM_SHARE, mse_share=MSE_SHARE,
    learning_rate=LEARNING_RATE, scheduler_factor=SCHEDULER_FACTOR,scheduler_patience=SCHEDULER_PATIENCE,
    max_epochs=MAX_EPOCHS, patience=PATIENCE, min_delta=MIN_DELTA, do_early_stopping=True,
    device=device, path=PATH, filename=FILENAME
)

In [ ]:
epochs_range = range(len(df_metrics_batchnorm["train_loss_vals"]))
plot_curves(epochs_range, df_metrics_batchnorm["train_loss_vals"], df_metrics_batchnorm["validation_loss_vals"], var_to_plot="Loss", filename="nn_training/loss_curves_batchnorm.png")
plot_curves(epochs_range, df_metrics_batchnorm["train_mae_vals"], df_metrics_batchnorm["validation_mae_vals"], var_to_plot="MAE", filename="nn_training/mae_curves_batchnorm.png")
plot_curves(epochs_range, df_metrics_batchnorm["train_psnr_vals"], df_metrics_batchnorm["validation_psnr_vals"], var_to_plot="PSNR", filename="nn_training/psnr_curves_batchnorm.png")
plot_curves(epochs_range, df_metrics_batchnorm["train_ssim_vals"], df_metrics_batchnorm["validation_ssim_vals"], var_to_plot="SSIM", filename="nn_training/ssim_curves_batchnorm.png")

In [ ]:
# -------------------- RESIDUAL NET -------------------------
SEED = 42
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.manual_seed(SEED)

# Create model instance and move it to device
residual_model = DeconvolutionUNet_Residual().to(device)
# Get number of trainable params
num_trainable_params = sum(
    p.numel() for p in residual_model.parameters() if p.requires_grad
)
print(f"Number of trainable parameters for model: {num_trainable_params}")

# Define loss function
MAE_SHARE = 1.0
EDGE_SHARE = 0.0
SSIM_SHARE = 0.0
MSE_SHARE = 0.0

# Define optimizer
LEARNING_RATE = 1e-4

# Define scheduler
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 3

# Early Stopping
MAX_EPOCHS = 200
PATIENCE = 15
MIN_DELTA = 1e-4

# Saving
PATH = "nn_training/models"
FILENAME = "nn_residual_model"

df_metrics_residual, df_about_residual = run_experiment(
    model=residual_model, seed=SEED, overlap=0.5,
    train_dataloader=train_dataloader, validation_dataloader=validation_dataloader, batch_size=BATCH_SIZE,
    mae_share=MAE_SHARE, edge_share=EDGE_SHARE, ssim_share=SSIM_SHARE, mse_share=MSE_SHARE,
    learning_rate=LEARNING_RATE, scheduler_factor=SCHEDULER_FACTOR,scheduler_patience=SCHEDULER_PATIENCE,
    max_epochs=MAX_EPOCHS, patience=PATIENCE, min_delta=MIN_DELTA, do_early_stopping=True,
    device=device, path=PATH, filename=FILENAME
)

In [ ]:
epochs_range = range(len(df_metrics_residual["train_loss_vals"]))
plot_curves(epochs_range, df_metrics_residual["train_loss_vals"], df_metrics_residual["validation_loss_vals"], var_to_plot="Loss", filename="nn_training/loss_curves_residual.png")
plot_curves(epochs_range, df_metrics_residual["train_mae_vals"], df_metrics_residual["validation_mae_vals"], var_to_plot="MAE", filename="nn_training/mae_curves_residual.png")
plot_curves(epochs_range, df_metrics_residual["train_psnr_vals"], df_metrics_residual["validation_psnr_vals"], var_to_plot="PSNR", filename="nn_training/psnr_curves_residual.png")
plot_curves(epochs_range, df_metrics_residual["train_ssim_vals"], df_metrics_residual["validation_ssim_vals"], var_to_plot="SSIM", filename="nn_training/ssim_curves_residual.png")

In [ ]:
# -------------------- HYBRID LOSS -------------------------
SEED = 42
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.manual_seed(SEED)

# Create model instance and move it to device
hybrid_model = DeconvolutionUNet_Residual().to(device)
# Get number of trainable params
num_trainable_params = sum(
    p.numel() for p in hybrid_model.parameters() if p.requires_grad
)
print(f"Number of trainable parameters for model: {num_trainable_params}")

# Define loss function
MAE_SHARE = 0.7
EDGE_SHARE = 0.0
SSIM_SHARE = 0.3
MSE_SHARE = 0.0

# Define optimizer
LEARNING_RATE = 1e-4

# Define scheduler
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 3

# Early Stopping
MAX_EPOCHS = 200
PATIENCE = 15
MIN_DELTA = 1e-4

# Saving
PATH = "nn_training/models"
FILENAME = "nn_hybrid_model"

df_metrics_hybrid, df_about_hybrid = run_experiment(
    model=hybrid_model, seed=SEED, overlap=0.5,
    train_dataloader=train_dataloader, validation_dataloader=validation_dataloader, batch_size=BATCH_SIZE,
    mae_share=MAE_SHARE, edge_share=EDGE_SHARE, ssim_share=SSIM_SHARE, mse_share=MSE_SHARE,
    learning_rate=LEARNING_RATE, scheduler_factor=SCHEDULER_FACTOR,scheduler_patience=SCHEDULER_PATIENCE,
    max_epochs=MAX_EPOCHS, patience=PATIENCE, min_delta=MIN_DELTA, do_early_stopping=True,
    device=device, path=PATH, filename=FILENAME
)

In [ ]:
epochs_range = range(len(df_metrics_hybrid["train_loss_vals"]))
plot_curves(epochs_range, df_metrics_hybrid["train_loss_vals"], df_metrics_hybrid["validation_loss_vals"], var_to_plot="Loss", filename="nn_training/loss_curves_hybrid.png")
plot_curves(epochs_range, df_metrics_hybrid["train_mae_vals"], df_metrics_hybrid["validation_mae_vals"], var_to_plot="MAE", filename="nn_training/mae_curves_hybrid.png")
plot_curves(epochs_range, df_metrics_hybrid["train_psnr_vals"], df_metrics_hybrid["validation_psnr_vals"], var_to_plot="PSNR", filename="nn_training/psnr_curves_hybrid.png")
plot_curves(epochs_range, df_metrics_hybrid["train_ssim_vals"], df_metrics_hybrid["validation_ssim_vals"], var_to_plot="SSIM", filename="nn_training/ssim_curves_hybrid.png")

In [ ]:
# ----------------- Model Comparison -----------------
model_metrics_files = {
    "Baseline": "nn_training/models/nn_baseline_model_metrics.csv",
    "BatchNorm": "nn_training/models/nn_batchnorm_model_metrics.csv",
    "Residual": "nn_training/models/nn_residual_model_metrics.csv",
    "Hybrid": "nn_training/models/nn_hybrid_model_metrics.csv"
}
validation_metrics_comparison(model_metrics_files, metric_name="validation_loss_vals", filename="nn_training/val_loss_comparison.png")
validation_metrics_comparison(model_metrics_files, metric_name="validation_mae_vals", filename="nn_training/val_mae_comparison.png")
validation_metrics_comparison(model_metrics_files, metric_name="validation_psnr_vals", filename="nn_training/val_psnr_comparison.png")
validation_metrics_comparison(model_metrics_files, metric_name="validation_ssim_vals", filename="nn_training/val_ssim_comparison.png")

In [ ]:
model_about_files = {
    "Baseline": "nn_training/models/nn_baseline_model_about.csv",
    "BatchNorm": "nn_training/models/nn_batchnorm_model_about.csv",
    "Residual": "nn_training/models/nn_residual_model_about.csv",
    "Hybrid": "nn_training/models/nn_hybrid_model_about.csv"
}

df = model_comparison_table(model_about_files)

# Create a nice table to display training results
display_df = df.copy()

float_cols = display_df.select_dtypes(include="float").columns
display_df[float_cols] = display_df[float_cols].map(lambda x: f"{x:.4f}")

fig, ax = plt.subplots(figsize=(8, 2))
ax.axis("off")

table = ax.table(
    cellText=display_df.values,
    colLabels=display_df.columns,
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.3, 2.0)

table.auto_set_column_width(col=list(range(len(df.columns))))

cmap = plt.get_cmap("cool")
num_models = len(df)

for r in range(1, len(df) + 1):
    color = list(cmap((r - 1) / (num_models - 1)))
    color[3] = 0.25  # Make the color 25% opaque
    for c in range(len(df.columns)):
        table[(r, c)].set_facecolor(color)

plt.savefig("nn_training/nn_comp_table.png", dpi=300, bbox_inches="tight")
plt.show()
